In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET: Same loading style as your working code
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []

    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue

    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)

        # Pre-allocate numpy arrays in RAM — same as your working code
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except:
                continue

            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue

            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue

            # normalize
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)

            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self):
        return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]          # (n, 4)

        # JEPA: split into two disjoint views
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)

        # pad to fixed 250 so DataLoader can stack without custom collate
        ctx_pad = torch.zeros(250, 4)
        tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool)
        tgt_mask = torch.zeros(250, dtype=torch.bool)

        ctx = pts[perm[:nc]]
        tgt = pts[perm[nc:]]

        ctx_pad[:ctx.size(0)] = ctx
        tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True
        tgt_mask[:tgt.size(0)] = True

        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. T-NET: Learned input alignment
# ==========================================
class TNet(nn.Module):
    """Learns a 4x4 transform on input points, initialized to identity."""
    def __init__(self, d=4, h=64):
        super().__init__()
        self.d = d
        self.phi = nn.Sequential(
            nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(),
        )
        self.fc = nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Linear(h, d * d))
        nn.init.zeros_(self.fc[-1].weight)
        nn.init.zeros_(self.fc[-1].bias)
        self.fc[-1].bias.data.copy_(torch.eye(d).flatten())  # init to identity

    def forward(self, x, mask=None):
        # x: (B, N, 4), mask: (B, N) bool
        h = self.phi(x)                                       # (B, N, h)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
            pooled = h.sum(1) / mask.sum(1, keepdim=True).clamp(1).float()
        else:
            pooled = h.mean(1)
        T = self.fc(pooled).view(-1, self.d, self.d)          # (B, 4, 4)
        x_aligned = torch.bmm(x, T)                           # (B, N, 4)
        return x_aligned, T

    @staticmethod
    def ortho_reg(T, w=0.001):
        """Regularize T toward orthogonal: ||T·T^T - I||^2"""
        I = torch.eye(T.size(1), device=T.device).unsqueeze(0)
        return w * (torch.bmm(T, T.transpose(1, 2)) - I).pow(2).sum((1, 2)).mean()


# ==========================================
# 3. ENCODER: T-Net + DeepSets φ + Cross-Attention K queries
# ==========================================
class ContextEncoder(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=256, latent_dim=128, K=8, n_heads=4, use_tnet=False):
        super().__init__()
        self.use_tnet = use_tnet
        self.K = K

        if use_tnet:
            self.tnet = TNet(input_dim, hidden_dim // 4)

        # DeepSets φ: shared MLP per point
        self.point_processor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        # K learnable query tokens
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)

        # Cross-attention: queries attend to point features
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim),
        )

    def forward(self, x, mask=None):
        """
        x:    (B, 250, 4)
        mask: (B, 250) bool, True = valid point
        Returns: (B, K, latent_dim) — K latent vectors
        Also returns T matrix if use_tnet for regularization
        """
        T_matrix = None

        # T-Net alignment
        if self.use_tnet:
            x, T_matrix = self.tnet(x, mask)

        # DeepSets φ per-point
        h = self.point_processor(x)                             # (B, 250, D)

        # Cross-attention: K queries read from point features
        q = self.queries.expand(x.size(0), -1, -1)             # (B, K, D)
        key_pad = ~mask if mask is not None else None
        attn_out = self.cross_attn(
            self.norm1(q), h, h, key_padding_mask=key_pad
        )[0]
        q = q + attn_out                                        # residual
        q = q + self.ffn(self.norm2(q))                         # FFN residual

        return q, T_matrix                                      # (B, K, D), (B, 4, 4) or None


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim),
            nn.LayerNorm(pred_dim),
            nn.GELU(),
            nn.Linear(pred_dim, dim),
        )

    def forward(self, z):
        return self.net(z)  # (B, K, D) → (B, K, D)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Get valid indices
    indices = get_valid_indices(npz_file)

    # 2. Load Dataset into RAM
    dataset = JEPADataset(npz_file, indices)

    # 3. Train/val split
    n_train = int(0.9 * len(dataset))
    n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    # 4. Online encoder + Target encoder (EMA copy) + Predictor
    context_encoder = ContextEncoder(K=K, use_tnet=use_tnet).to(device)
    target_encoder  = ContextEncoder(K=K, use_tnet=use_tnet).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    # Target = copy of context, frozen
    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs
    )

    total_steps = epochs * len(train_loader)
    global_step = 0
    best_val = float("inf")

    tnet_tag = "T-Net ON" if use_tnet else "T-Net OFF"
    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nJEPA Training on {device} | K={K} | {tnet_tag} | {n_params:,} encoder params")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        # ── Train ──
        context_encoder.train()
        predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx      = ctx.to(device, non_blocking=True)
            tgt      = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            # Online encoder → context view → K vectors
            z_context, T_ctx = context_encoder(ctx, ctx_mask)

            # Target encoder → target view → K vectors (no grad!)
            with torch.no_grad():
                z_target, _ = target_encoder(tgt, tgt_mask)

            # Predictor: context K vectors → predicted target K vectors
            z_predicted = predictor(z_context)

            # Loss in normalized latent space
            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)

            loss_inv = F.smooth_l1_loss(pred_n, tgt_n.detach())
            loss_var = F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            loss = loss_inv + loss_var

            # T-Net orthogonality regularization
            if use_tnet and T_ctx is not None:
                loss = loss + TNet.ortho_reg(T_ctx)

            if torch.isnan(loss):
                loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(context_encoder.parameters()) + list(predictor.parameters()), 1.0
            )
            optimizer.step()
            scheduler.step()

            # EMA update target encoder
            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)

            with torch.no_grad():
                cos = F.cosine_similarity(pred_n.flatten(0, 1), tgt_n.flatten(0, 1), dim=-1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item()
            total_cos  += cos.item()
            total_std  += std.item()
            n_batches  += 1
            global_step += 1

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        # ── Validate ──
        context_encoder.eval()
        predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0

        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx      = ctx.to(device)
                tgt      = tgt.to(device)
                ctx_mask = ctx_mask.to(device)
                tgt_mask = tgt_mask.to(device)

                z_c, _ = context_encoder(ctx, ctx_mask)
                z_t, _ = target_encoder(tgt, tgt_mask)
                z_p    = predictor(z_c)

                pn = F.normalize(z_p, dim=-1, eps=1e-6)
                tn = F.normalize(z_t, dim=-1, eps=1e-6)

                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0, 1), tn.flatten(0, 1), dim=-1).mean()

                val_loss += vl.item()
                val_cos  += vc.item()
                val_n    += 1

        # ── Log ──
        tl = total_loss / n_batches
        tc = total_cos / n_batches
        ts = total_std / n_batches
        vl = val_loss / val_n
        vc = val_cos / val_n

        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

        if vl < best_val:
            best_val = vl

    # Save at the end of training
    torch.save({
        "epoch": epochs,
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val,
        "K": K, "use_tnet": use_tnet,
    }, "jepa_best.pt")
    print(f"\nDone! Best val_loss={best_val:.4f} | Saved jepa_best.pt")


# ==========================================
# 6. T-NET ABLATION: run both, compare
# ==========================================
def ablation(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3):
    print("=" * 60)
    print("  ABLATION: WITHOUT T-Net")
    print("=" * 60)
    train_jepa(npz_file, epochs, K, batch_size, lr, use_tnet=False)

    print("\n" + "=" * 60)
    print("  ABLATION: WITH T-Net")
    print("=" * 60)
    train_jepa(npz_file, epochs, K, batch_size, lr, use_tnet=True)


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50,
    K=8,
    batch_size=1024,
    lr=2e-3,
    use_tnet=False,   # set True to enable T-Net, or call ablation() for both
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:05<00:00, 241.26it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

JEPA Training on cuda | K=8 | T-Net OFF | 234,752 encoder params
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:08<00:00,  4.58it/s, loss=0.9269, cos=0.004]


  loss=0.9386 cos=-0.003 std=0.069 | val=0.9266 val_cos=0.006


...

  loss=0.9123 cos=0.919 std=0.088 | val=0.9123 val_cos=0.919


Epoch 49/50: 100%|██████████| 39/39 [00:08<00:00,  4.36it/s, loss=0.9123, cos=0.919]


  loss=0.9123 cos=0.919 std=0.088 | val=0.9123 val_cos=0.919


Epoch 50/50: 100%|██████████| 39/39 [00:09<00:00,  4.29it/s, loss=0.9123, cos=0.919]


  loss=0.9123 cos=0.919 std=0.088 | val=0.9123 val_cos=0.919

Done! Best val_loss=0.9123 | Saved jepa_best.pt


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET: Same loading style as your working code
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []

    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue

    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)

        # Pre-allocate numpy arrays in RAM — same as your working code
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except:
                continue

            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue

            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue

            # normalize
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)

            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self):
        return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]          # (n, 4)

        # JEPA: split into two disjoint views
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)

        # pad to fixed 250 so DataLoader can stack without custom collate
        ctx_pad = torch.zeros(250, 4)
        tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool)
        tgt_mask = torch.zeros(250, dtype=torch.bool)

        ctx = pts[perm[:nc]]
        tgt = pts[perm[nc:]]

        ctx_pad[:ctx.size(0)] = ctx
        tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True
        tgt_mask[:tgt.size(0)] = True

        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. T-NET: Learned input alignment
# ==========================================
class TNet(nn.Module):
    """Learns a 4x4 transform on input points, initialized to identity."""
    def __init__(self, d=4, h=64):
        super().__init__()
        self.d = d
        self.phi = nn.Sequential(
            nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(),
        )
        self.fc = nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Linear(h, d * d))
        nn.init.zeros_(self.fc[-1].weight)
        nn.init.zeros_(self.fc[-1].bias)
        self.fc[-1].bias.data.copy_(torch.eye(d).flatten())  # init to identity

    def forward(self, x, mask=None):
        # x: (B, N, 4), mask: (B, N) bool
        h = self.phi(x)                                       # (B, N, h)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
            pooled = h.sum(1) / mask.sum(1, keepdim=True).clamp(1).float()
        else:
            pooled = h.mean(1)
        T = self.fc(pooled).view(-1, self.d, self.d)          # (B, 4, 4)
        x_aligned = torch.bmm(x, T)                           # (B, N, 4)
        return x_aligned, T

    @staticmethod
    def ortho_reg(T, w=0.001):
        """Regularize T toward orthogonal: ||T·T^T - I||^2"""
        I = torch.eye(T.size(1), device=T.device).unsqueeze(0)
        return w * (torch.bmm(T, T.transpose(1, 2)) - I).pow(2).sum((1, 2)).mean()


# ==========================================
# 3. ENCODER: T-Net + DeepSets φ + Cross-Attention K queries
# ==========================================
class ContextEncoder(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=256, latent_dim=128, K=8, n_heads=4, use_tnet=False):
        super().__init__()
        self.use_tnet = use_tnet
        self.K = K

        if use_tnet:
            self.tnet = TNet(input_dim, hidden_dim // 4)

        # DeepSets φ: shared MLP per point
        self.point_processor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        # K learnable query tokens
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)

        # Cross-attention: queries attend to point features
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim),
        )

    def forward(self, x, mask=None):
        """
        x:    (B, 250, 4)
        mask: (B, 250) bool, True = valid point
        Returns: (B, K, latent_dim) — K latent vectors
        Also returns T matrix if use_tnet for regularization
        """
        T_matrix = None

        # T-Net alignment
        if self.use_tnet:
            x, T_matrix = self.tnet(x, mask)

        # DeepSets φ per-point
        h = self.point_processor(x)                             # (B, 250, D)

        # Cross-attention: K queries read from point features
        q = self.queries.expand(x.size(0), -1, -1)             # (B, K, D)
        key_pad = ~mask if mask is not None else None
        attn_out = self.cross_attn(
            self.norm1(q), h, h, key_padding_mask=key_pad
        )[0]
        q = q + attn_out                                        # residual
        q = q + self.ffn(self.norm2(q))                         # FFN residual

        return q, T_matrix                                      # (B, K, D), (B, 4, 4) or None


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim),
            nn.LayerNorm(pred_dim),
            nn.GELU(),
            nn.Linear(pred_dim, dim),
        )

    def forward(self, z):
        return self.net(z)  # (B, K, D) → (B, K, D)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Get valid indices
    indices = get_valid_indices(npz_file)

    # 2. Load Dataset into RAM
    dataset = JEPADataset(npz_file, indices)

    # 3. Train/val split
    n_train = int(0.9 * len(dataset))
    n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    # 4. Online encoder + Target encoder (EMA copy) + Predictor
    context_encoder = ContextEncoder(K=K, use_tnet=use_tnet).to(device)
    target_encoder  = ContextEncoder(K=K, use_tnet=use_tnet).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    # Target = copy of context, frozen
    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs
    )

    total_steps = epochs * len(train_loader)
    global_step = 0
    best_val = float("inf")

    tnet_tag = "T-Net ON" if use_tnet else "T-Net OFF"
    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nJEPA Training on {device} | K={K} | {tnet_tag} | {n_params:,} encoder params")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        # ── Train ──
        context_encoder.train()
        predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx      = ctx.to(device, non_blocking=True)
            tgt      = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            # Online encoder → context view → K vectors
            z_context, T_ctx = context_encoder(ctx, ctx_mask)

            # Target encoder → target view → K vectors (no grad!)
            with torch.no_grad():
                z_target, _ = target_encoder(tgt, tgt_mask)

            # Predictor: context K vectors → predicted target K vectors
            z_predicted = predictor(z_context)

            # Loss in normalized latent space
            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)

            loss_inv = F.smooth_l1_loss(pred_n, tgt_n.detach())
            loss_var = F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            loss = loss_inv + loss_var

            # T-Net orthogonality regularization
            if use_tnet and T_ctx is not None:
                loss = loss + TNet.ortho_reg(T_ctx)

            if torch.isnan(loss):
                loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(context_encoder.parameters()) + list(predictor.parameters()), 1.0
            )
            optimizer.step()
            scheduler.step()

            # EMA update target encoder
            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)

            with torch.no_grad():
                cos = F.cosine_similarity(pred_n.flatten(0, 1), tgt_n.flatten(0, 1), dim=-1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item()
            total_cos  += cos.item()
            total_std  += std.item()
            n_batches  += 1
            global_step += 1

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        # ── Validate ──
        context_encoder.eval()
        predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0

        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx      = ctx.to(device)
                tgt      = tgt.to(device)
                ctx_mask = ctx_mask.to(device)
                tgt_mask = tgt_mask.to(device)

                z_c, _ = context_encoder(ctx, ctx_mask)
                z_t, _ = target_encoder(tgt, tgt_mask)
                z_p    = predictor(z_c)

                pn = F.normalize(z_p, dim=-1, eps=1e-6)
                tn = F.normalize(z_t, dim=-1, eps=1e-6)

                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0, 1), tn.flatten(0, 1), dim=-1).mean()

                val_loss += vl.item()
                val_cos  += vc.item()
                val_n    += 1

        # ── Log ──
        tl = total_loss / n_batches
        tc = total_cos / n_batches
        ts = total_std / n_batches
        vl = val_loss / val_n
        vc = val_cos / val_n

        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

        if vl < best_val:
            best_val = vl

    # Save at the end of training
    torch.save({
        "epoch": epochs,
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val,
        "K": K, "use_tnet": use_tnet,
    }, "jepa_best.pt")
    print(f"\nDone! Best val_loss={best_val:.4f} | Saved jepa_best.pt")


# ==========================================
# 6. T-NET ABLATION: run both, compare
# ==========================================
def ablation(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3):
    print("=" * 60)
    print("  ABLATION: WITHOUT T-Net")
    print("=" * 60)
    train_jepa(npz_file, epochs, K, batch_size, lr, use_tnet=False)

    print("\n" + "=" * 60)
    print("  ABLATION: WITH T-Net")
    print("=" * 60)
    train_jepa(npz_file, epochs, K, batch_size, lr, use_tnet=True)


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50,
    K=8,
    batch_size=1024,
    lr=2e-3,
    use_tnet=True,   # set True to enable T-Net, or call ablation() for both
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:02<00:00, 245.34it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

JEPA Training on cuda | K=8 | T-Net ON | 244,688 encoder params
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:13<00:00,  2.88it/s, loss=0.9272, cos=0.007]


  loss=0.9404 cos=0.001 std=0.067 | val=0.9269 val_cos=0.002


...

  loss=0.9124 cos=0.907 std=0.088 | val=0.9124 val_cos=0.907


Epoch 49/50: 100%|██████████| 39/39 [00:11<00:00,  3.34it/s, loss=0.9124, cos=0.907]


  loss=0.9124 cos=0.907 std=0.088 | val=0.9124 val_cos=0.907


Epoch 50/50: 100%|██████████| 39/39 [00:11<00:00,  3.36it/s, loss=0.9124, cos=0.908]


  loss=0.9124 cos=0.908 std=0.088 | val=0.9124 val_cos=0.908

Done! Best val_loss=0.9124 | Saved jepa_best.pt


In [ ]:
!mv jepa_best.pt /content/drive/MyDrive/symbolic_data/Architecture_tests_17march/

/content/drive/MyDrive/symbolic_data/Architecture_tests_17march/Deepsets_tnet_jepa_best.pt

In [ ]:
"""
LM-JEPA: Set Transformer + T-Net Encoder
Points self-attend first, THEN K queries cross-attend.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []
    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue
    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except: continue
            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue
            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self): return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)
        ctx_pad = torch.zeros(250, 4); tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool); tgt_mask = torch.zeros(250, dtype=torch.bool)
        ctx = pts[perm[:nc]]; tgt = pts[perm[nc:]]
        ctx_pad[:ctx.size(0)] = ctx; tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True; tgt_mask[:tgt.size(0)] = True
        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. T-NET
# ==========================================
class TNet(nn.Module):
    def __init__(self, d=4, h=64):
        super().__init__()
        self.d = d
        self.phi = nn.Sequential(
            nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU())
        self.fc = nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Linear(h, d * d))
        nn.init.zeros_(self.fc[-1].weight); nn.init.zeros_(self.fc[-1].bias)
        self.fc[-1].bias.data.copy_(torch.eye(d).flatten())

    def forward(self, x, mask=None):
        h = self.phi(x)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
            pooled = h.sum(1) / mask.sum(1, keepdim=True).clamp(1).float()
        else: pooled = h.mean(1)
        T = self.fc(pooled).view(-1, self.d, self.d)
        return torch.bmm(x, T), T

    @staticmethod
    def ortho_reg(T, w=0.001):
        I = torch.eye(T.size(1), device=T.device).unsqueeze(0)
        return w * (torch.bmm(T, T.transpose(1, 2)) - I).pow(2).sum((1, 2)).mean()


# ==========================================
# 3. SET TRANSFORMER ENCODER
# ==========================================
class SetTransformerEncoder(nn.Module):
    """
    Points self-attend (learn inter-point relationships),
    THEN K queries cross-attend to enriched features.
    """
    def __init__(self, input_dim=4, hidden_dim=128, latent_dim=128,
                 K=8, n_heads=4, n_self_layers=2, use_tnet=False):
        super().__init__()
        self.use_tnet = use_tnet
        if use_tnet: self.tnet = TNet(input_dim, 64)

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

        # Self-attention among points
        self.self_attn_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=hidden_dim, nhead=n_heads,
                dim_feedforward=hidden_dim * 2, dropout=0.1,
                activation="gelu", batch_first=True, norm_first=True)
            for _ in range(n_self_layers)])

        self.proj = nn.Linear(hidden_dim, latent_dim) if hidden_dim != latent_dim else nn.Identity()

        # K queries cross-attend to enriched points
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))

    def forward(self, x, mask=None):
        T_matrix = None
        if self.use_tnet: x, T_matrix = self.tnet(x, mask)

        h = self.input_proj(x)
        key_pad = ~mask if mask is not None else None
        for layer in self.self_attn_layers:
            h = layer(h, src_key_padding_mask=key_pad)

        h = self.proj(h)

        q = self.queries.expand(x.size(0), -1, -1)
        attn_out = self.cross_attn(self.norm1(q), h, h, key_padding_mask=key_pad)[0]
        q = q + attn_out
        q = q + self.ffn(self.norm2(q))
        return q, T_matrix


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim), nn.LayerNorm(pred_dim), nn.GELU(),
            nn.Linear(pred_dim, dim))
    def forward(self, z): return self.net(z)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    indices = get_valid_indices(npz_file)
    dataset = JEPADataset(npz_file, indices)
    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    context_encoder = SetTransformerEncoder(K=K, use_tnet=use_tnet).to(device)
    target_encoder  = SetTransformerEncoder(K=K, use_tnet=use_tnet).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters(): p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)

    total_steps = epochs * len(train_loader)
    global_step = 0; best_val = float("inf")

    tnet_tag = "+T-Net" if use_tnet else ""
    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nSET TRANSFORMER{tnet_tag} | K={K} | {n_params:,} params | {device}")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        context_encoder.train(); predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx = ctx.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            z_context, T_ctx = context_encoder(ctx, ctx_mask)
            with torch.no_grad(): z_target, _ = target_encoder(tgt, tgt_mask)
            z_predicted = predictor(z_context)

            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)
            loss = F.smooth_l1_loss(pred_n, tgt_n.detach()) + F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            if use_tnet and T_ctx is not None: loss = loss + TNet.ortho_reg(T_ctx)
            if torch.isnan(loss): loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(list(context_encoder.parameters()) + list(predictor.parameters()), 1.0)
            optimizer.step(); scheduler.step()

            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)
                cos = F.cosine_similarity(pred_n.flatten(0,1), tgt_n.flatten(0,1), -1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item(); total_cos += cos.item()
            total_std += std.item(); n_batches += 1; global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        context_encoder.eval(); predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0
        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx = ctx.to(device); tgt = tgt.to(device)
                ctx_mask = ctx_mask.to(device); tgt_mask = tgt_mask.to(device)
                z_c, _ = context_encoder(ctx, ctx_mask)
                z_t, _ = target_encoder(tgt, tgt_mask)
                z_p = predictor(z_c)
                pn = F.normalize(z_p, -1, eps=1e-6); tn = F.normalize(z_t, -1, eps=1e-6)
                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0,1), tn.flatten(0,1), -1).mean()
                val_loss += vl.item(); val_cos += vc.item(); val_n += 1

        tl = total_loss/n_batches; tc = total_cos/n_batches; ts = total_std/n_batches
        vl = val_loss/val_n; vc = val_cos/val_n
        if vl < best_val: best_val = vl
        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

    torch.save({
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val, "K": K, "use_tnet": use_tnet,
    }, f"/content/drive/MyDrive/Colab Notebooks/symbolic_data/Architecture_tests_17march/jepa_set_transformer{'_tnet' if use_tnet else ''}.pt")
    print(f"\nDone! SET TRANSFORMER{tnet_tag} — best val={best_val:.4f}")


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False,
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:21<00:00, 221.99it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

SET TRANSFORMER | K=8 | 399,360 params | cuda
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:28<00:00,  1.39it/s, loss=0.9256, cos=0.002]


  loss=0.9361 cos=-0.033 std=0.072 | val=27.2829 val_cos=0.007


...

  loss=0.9123 cos=0.914 std=0.088 | val=4.3335 val_cos=0.551


Epoch 49/50: 100%|██████████| 39/39 [00:31<00:00,  1.24it/s, loss=0.9123, cos=0.914]


  loss=0.9123 cos=0.914 std=0.088 | val=3.7427 val_cos=0.557


Epoch 50/50: 100%|██████████| 39/39 [00:31<00:00,  1.23it/s, loss=0.9124, cos=0.914]


  loss=0.9123 cos=0.914 std=0.088 | val=4.3132 val_cos=0.551

Done! SET TRANSFORMER — best val=3.6399


In [3]:
"""
LM-JEPA: Set Transformer + T-Net Encoder
Points self-attend first, THEN K queries cross-attend.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []
    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue
    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except: continue
            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue
            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self): return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)
        ctx_pad = torch.zeros(250, 4); tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool); tgt_mask = torch.zeros(250, dtype=torch.bool)
        ctx = pts[perm[:nc]]; tgt = pts[perm[nc:]]
        ctx_pad[:ctx.size(0)] = ctx; tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True; tgt_mask[:tgt.size(0)] = True
        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. T-NET
# ==========================================
class TNet(nn.Module):
    def __init__(self, d=4, h=64):
        super().__init__()
        self.d = d
        self.phi = nn.Sequential(
            nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU())
        self.fc = nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Linear(h, d * d))
        nn.init.zeros_(self.fc[-1].weight); nn.init.zeros_(self.fc[-1].bias)
        self.fc[-1].bias.data.copy_(torch.eye(d).flatten())

    def forward(self, x, mask=None):
        h = self.phi(x)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
            pooled = h.sum(1) / mask.sum(1, keepdim=True).clamp(1).float()
        else: pooled = h.mean(1)
        T = self.fc(pooled).view(-1, self.d, self.d)
        return torch.bmm(x, T), T

    @staticmethod
    def ortho_reg(T, w=0.001):
        I = torch.eye(T.size(1), device=T.device).unsqueeze(0)
        return w * (torch.bmm(T, T.transpose(1, 2)) - I).pow(2).sum((1, 2)).mean()


# ==========================================
# 3. SET TRANSFORMER ENCODER
# ==========================================
class SetTransformerEncoder(nn.Module):
    """
    Points self-attend (learn inter-point relationships),
    THEN K queries cross-attend to enriched features.
    """
    def __init__(self, input_dim=4, hidden_dim=128, latent_dim=128,
                 K=8, n_heads=4, n_self_layers=2, use_tnet=False):
        super().__init__()
        self.use_tnet = use_tnet
        if use_tnet: self.tnet = TNet(input_dim, 64)

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

        # Self-attention among points
        self.self_attn_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=hidden_dim, nhead=n_heads,
                dim_feedforward=hidden_dim * 2, dropout=0.1,
                activation="gelu", batch_first=True, norm_first=True)
            for _ in range(n_self_layers)])

        self.proj = nn.Linear(hidden_dim, latent_dim) if hidden_dim != latent_dim else nn.Identity()

        # K queries cross-attend to enriched points
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(latent_dim)
        self.norm2 = nn.LayerNorm(latent_dim)
        self.ffn = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
            nn.Linear(latent_dim * 2, latent_dim))

    def forward(self, x, mask=None):
        T_matrix = None
        if self.use_tnet: x, T_matrix = self.tnet(x, mask)

        h = self.input_proj(x)
        key_pad = ~mask if mask is not None else None
        for layer in self.self_attn_layers:
            h = layer(h, src_key_padding_mask=key_pad)

        h = self.proj(h)

        q = self.queries.expand(x.size(0), -1, -1)
        attn_out = self.cross_attn(self.norm1(q), h, h, key_padding_mask=key_pad)[0]
        q = q + attn_out
        q = q + self.ffn(self.norm2(q))
        return q, T_matrix


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim), nn.LayerNorm(pred_dim), nn.GELU(),
            nn.Linear(pred_dim, dim))
    def forward(self, z): return self.net(z)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    indices = get_valid_indices(npz_file)
    dataset = JEPADataset(npz_file, indices)
    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    context_encoder = SetTransformerEncoder(K=K, use_tnet=use_tnet).to(device)
    target_encoder  = SetTransformerEncoder(K=K, use_tnet=use_tnet).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters(): p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)

    total_steps = epochs * len(train_loader)
    global_step = 0; best_val = float("inf")

    tnet_tag = "+T-Net" if use_tnet else ""
    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nSET TRANSFORMER{tnet_tag} | K={K} | {n_params:,} params | {device}")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        context_encoder.train(); predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx = ctx.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            z_context, T_ctx = context_encoder(ctx, ctx_mask)
            with torch.no_grad(): z_target, _ = target_encoder(tgt, tgt_mask)
            z_predicted = predictor(z_context)

            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)
            loss = F.smooth_l1_loss(pred_n, tgt_n.detach()) + F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            if use_tnet and T_ctx is not None: loss = loss + TNet.ortho_reg(T_ctx)
            if torch.isnan(loss): loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(list(context_encoder.parameters()) + list(predictor.parameters()), 1.0)
            optimizer.step(); scheduler.step()

            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)
                cos = F.cosine_similarity(pred_n.flatten(0,1), tgt_n.flatten(0,1), -1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item(); total_cos += cos.item()
            total_std += std.item(); n_batches += 1; global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        context_encoder.eval(); predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0
        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx = ctx.to(device); tgt = tgt.to(device)
                ctx_mask = ctx_mask.to(device); tgt_mask = tgt_mask.to(device)
                z_c, _ = context_encoder(ctx, ctx_mask)
                z_t, _ = target_encoder(tgt, tgt_mask)
                z_p = predictor(z_c)
                pn = F.normalize(z_p, -1, eps=1e-6); tn = F.normalize(z_t, -1, eps=1e-6)
                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0,1), tn.flatten(0,1), -1).mean()
                val_loss += vl.item(); val_cos += vc.item(); val_n += 1

        tl = total_loss/n_batches; tc = total_cos/n_batches; ts = total_std/n_batches
        vl = val_loss/val_n; vc = val_cos/val_n
        if vl < best_val: best_val = vl
        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

    torch.save({
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val, "K": K, "use_tnet": use_tnet,
    }, f"/content/drive/MyDrive/Colab Notebooks/symbolic_data/Architecture_tests_17march/jepa_set_transformer{'_tnet' if use_tnet else ''}.pt")
    print(f"\nDone! SET TRANSFORMER{tnet_tag} — best val={best_val:.4f}")


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=True,
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:03<00:00, 243.77it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

SET TRANSFORMER+T-Net | K=8 | 409,296 params | cuda
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:29<00:00,  1.32it/s, loss=0.9260, cos=0.026]


  loss=0.9398 cos=-0.003 std=0.068 | val=24.3244 val_cos=-0.012


...

  loss=0.9123 cos=0.919 std=0.088 | val=4.5583 val_cos=0.551


Epoch 49/50: 100%|██████████| 39/39 [00:32<00:00,  1.21it/s, loss=0.9123, cos=0.919]


  loss=0.9123 cos=0.919 std=0.088 | val=4.5229 val_cos=0.544


Epoch 50/50: 100%|██████████| 39/39 [00:32<00:00,  1.21it/s, loss=0.9123, cos=0.918]


  loss=0.9123 cos=0.919 std=0.088 | val=4.3574 val_cos=0.545

Done! SET TRANSFORMER+T-Net — best val=4.0855


In [ ]:
"""
LM-JEPA: Perceiver Encoder (Multi-Round Cross-Attention)
3 rounds of: queries ← cross-attn(points) → self-attn(queries)
Each round refines what queries attend to.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []
    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue
    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except: continue
            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue
            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self): return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)
        ctx_pad = torch.zeros(250, 4); tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool); tgt_mask = torch.zeros(250, dtype=torch.bool)
        ctx = pts[perm[:nc]]; tgt = pts[perm[nc:]]
        ctx_pad[:ctx.size(0)] = ctx; tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True; tgt_mask[:tgt.size(0)] = True
        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. T-NET
# ==========================================
class TNet(nn.Module):
    def __init__(self, d=4, h=64):
        super().__init__()
        self.d = d
        self.phi = nn.Sequential(
            nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU())
        self.fc = nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Linear(h, d * d))
        nn.init.zeros_(self.fc[-1].weight); nn.init.zeros_(self.fc[-1].bias)
        self.fc[-1].bias.data.copy_(torch.eye(d).flatten())

    def forward(self, x, mask=None):
        h = self.phi(x)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
            pooled = h.sum(1) / mask.sum(1, keepdim=True).clamp(1).float()
        else: pooled = h.mean(1)
        T = self.fc(pooled).view(-1, self.d, self.d)
        return torch.bmm(x, T), T

    @staticmethod
    def ortho_reg(T, w=0.001):
        I = torch.eye(T.size(1), device=T.device).unsqueeze(0)
        return w * (torch.bmm(T, T.transpose(1, 2)) - I).pow(2).sum((1, 2)).mean()


# ==========================================
# 3. PERCEIVER ENCODER
# ==========================================
class PerceiverEncoder(nn.Module):
    """
    Multi-round Perceiver: each round =
      cross-attn (K queries read from N points) +
      self-attn  (K queries talk to each other)
    Round 1: coarse ("probably polynomial")
    Round 2: refine ("degree 3, positive leading coeff")
    Round 3: details ("coefficient near 2.7")
    """
    def __init__(self, input_dim=4, hidden_dim=128, latent_dim=128,
                 K=8, n_heads=4, n_rounds=3, use_tnet=False):
        super().__init__()
        self.use_tnet = use_tnet
        self.n_rounds = n_rounds
        if use_tnet: self.tnet = TNet(input_dim, 64)

        # per-point encoder
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, latent_dim), nn.LayerNorm(latent_dim))

        # K learnable queries
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)

        # Per-round layers
        self.cross_attns = nn.ModuleList()
        self.cross_norms = nn.ModuleList()
        self.cross_ffns  = nn.ModuleList()
        self.cross_fn    = nn.ModuleList()
        self.self_attns  = nn.ModuleList()
        self.self_norms  = nn.ModuleList()
        self.self_ffns   = nn.ModuleList()
        self.self_fn     = nn.ModuleList()

        for _ in range(n_rounds):
            self.cross_attns.append(nn.MultiheadAttention(latent_dim, n_heads, batch_first=True, dropout=0.2))
            self.cross_norms.append(nn.LayerNorm(latent_dim))
            self.cross_ffns.append(nn.Sequential(
                nn.Linear(latent_dim, latent_dim*2), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(latent_dim*2, latent_dim)))
            self.cross_fn.append(nn.LayerNorm(latent_dim))

            self.self_attns.append(nn.MultiheadAttention(latent_dim, n_heads, batch_first=True, dropout=0.2))
            self.self_norms.append(nn.LayerNorm(latent_dim))
            self.self_ffns.append(nn.Sequential(
                nn.Linear(latent_dim, latent_dim*2), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(latent_dim*2, latent_dim)))
            self.self_fn.append(nn.LayerNorm(latent_dim))

    def forward(self, x, mask=None):
        T_matrix = None
        if self.use_tnet: x, T_matrix = self.tnet(x, mask)

        h = self.phi(x)                                          # (B, N, D)
        q = self.queries.expand(x.size(0), -1, -1)               # (B, K, D)
        key_pad = ~mask if mask is not None else None

        for i in range(self.n_rounds):
            # cross-attention: queries ← points
            ca = self.cross_attns[i](self.cross_norms[i](q), h, h, key_padding_mask=key_pad)[0]
            q = q + ca
            q = q + self.cross_ffns[i](self.cross_fn[i](q))

            # self-attention: queries ↔ queries
            sa = self.self_attns[i](self.self_norms[i](q), q, q)[0]
            q = q + sa
            q = q + self.self_ffns[i](self.self_fn[i](q))

        return q, T_matrix                                       # (B, K, D)


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim), nn.LayerNorm(pred_dim), nn.GELU(),
            nn.Linear(pred_dim, dim))
    def forward(self, z): return self.net(z)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False, n_rounds=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    indices = get_valid_indices(npz_file)
    dataset = JEPADataset(npz_file, indices)
    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    context_encoder = PerceiverEncoder(K=K, use_tnet=use_tnet, n_rounds=n_rounds).to(device)
    target_encoder  = PerceiverEncoder(K=K, use_tnet=use_tnet, n_rounds=n_rounds).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters(): p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)

    total_steps = epochs * len(train_loader)
    global_step = 0; best_val = float("inf")

    tnet_tag = "+T-Net" if use_tnet else ""
    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nPERCEIVER{tnet_tag} | K={K} | rounds={n_rounds} | {n_params:,} params | {device}")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        context_encoder.train(); predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx = ctx.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            z_context, T_ctx = context_encoder(ctx, ctx_mask)
            with torch.no_grad(): z_target, _ = target_encoder(tgt, tgt_mask)
            z_predicted = predictor(z_context)

            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)
            loss = F.smooth_l1_loss(pred_n, tgt_n.detach()) + F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            if use_tnet and T_ctx is not None: loss = loss + TNet.ortho_reg(T_ctx)
            if torch.isnan(loss): loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(list(context_encoder.parameters()) + list(predictor.parameters()), 1.0)
            optimizer.step(); scheduler.step()

            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)
                cos = F.cosine_similarity(pred_n.flatten(0,1), tgt_n.flatten(0,1), -1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item(); total_cos += cos.item()
            total_std += std.item(); n_batches += 1; global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        context_encoder.eval(); predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0
        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx = ctx.to(device); tgt = tgt.to(device)
                ctx_mask = ctx_mask.to(device); tgt_mask = tgt_mask.to(device)
                z_c, _ = context_encoder(ctx, ctx_mask)
                z_t, _ = target_encoder(tgt, tgt_mask)
                z_p = predictor(z_c)
                pn = F.normalize(z_p, -1, eps=1e-6); tn = F.normalize(z_t, -1, eps=1e-6)
                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0,1), tn.flatten(0,1), -1).mean()
                val_loss += vl.item(); val_cos += vc.item(); val_n += 1

        tl = total_loss/n_batches; tc = total_cos/n_batches; ts = total_std/n_batches
        vl = val_loss/val_n; vc = val_cos/val_n
        if vl < best_val: best_val = vl
        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

    torch.save({
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val, "K": K, "use_tnet": use_tnet, "n_rounds": n_rounds,
    }, f"jepa_perceiver{'_tnet' if use_tnet else ''}.pt")
    print(f"\nDone! PERCEIVER{tnet_tag} — best val={best_val:.4f}")


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False, n_rounds=2,
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:03<00:00, 244.30it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

PERCEIVER | K=8 | rounds=2 | 548,608 params | cuda
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:10<00:00,  3.73it/s, loss=0.9257, cos=-0.029]


  loss=0.9346 cos=-0.054 std=0.074 | val=50.4190 val_cos=-0.002


...

  loss=0.9124 cos=0.903 std=0.088 | val=12.6635 val_cos=0.336


Epoch 49/50: 100%|██████████| 39/39 [00:09<00:00,  3.93it/s, loss=0.9124, cos=0.903]


  loss=0.9124 cos=0.903 std=0.088 | val=12.2180 val_cos=0.341


Epoch 50/50: 100%|██████████| 39/39 [00:09<00:00,  4.01it/s, loss=0.9124, cos=0.903]


  loss=0.9124 cos=0.903 std=0.088 | val=12.0183 val_cos=0.339

Done! PERCEIVER — best val=11.5980


In [4]:
"""
LM-JEPA: Perceiver Encoder (Multi-Round Cross-Attention)
3 rounds of: queries ← cross-attn(points) → self-attn(queries)
Each round refines what queries attend to.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []
    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue
    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except: continue
            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue
            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self): return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)
        ctx_pad = torch.zeros(250, 4); tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool); tgt_mask = torch.zeros(250, dtype=torch.bool)
        ctx = pts[perm[:nc]]; tgt = pts[perm[nc:]]
        ctx_pad[:ctx.size(0)] = ctx; tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True; tgt_mask[:tgt.size(0)] = True
        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. T-NET
# ==========================================
class TNet(nn.Module):
    def __init__(self, d=4, h=64):
        super().__init__()
        self.d = d
        self.phi = nn.Sequential(
            nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU())
        self.fc = nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Linear(h, d * d))
        nn.init.zeros_(self.fc[-1].weight); nn.init.zeros_(self.fc[-1].bias)
        self.fc[-1].bias.data.copy_(torch.eye(d).flatten())

    def forward(self, x, mask=None):
        h = self.phi(x)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
            pooled = h.sum(1) / mask.sum(1, keepdim=True).clamp(1).float()
        else: pooled = h.mean(1)
        T = self.fc(pooled).view(-1, self.d, self.d)
        return torch.bmm(x, T), T

    @staticmethod
    def ortho_reg(T, w=0.001):
        I = torch.eye(T.size(1), device=T.device).unsqueeze(0)
        return w * (torch.bmm(T, T.transpose(1, 2)) - I).pow(2).sum((1, 2)).mean()


# ==========================================
# 3. PERCEIVER ENCODER
# ==========================================
class PerceiverEncoder(nn.Module):
    """
    Multi-round Perceiver: each round =
      cross-attn (K queries read from N points) +
      self-attn  (K queries talk to each other)
    Round 1: coarse ("probably polynomial")
    Round 2: refine ("degree 3, positive leading coeff")
    Round 3: details ("coefficient near 2.7")
    """
    def __init__(self, input_dim=4, hidden_dim=128, latent_dim=128,
                 K=8, n_heads=4, n_rounds=3, use_tnet=False):
        super().__init__()
        self.use_tnet = use_tnet
        self.n_rounds = n_rounds
        if use_tnet: self.tnet = TNet(input_dim, 64)

        # per-point encoder
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, latent_dim), nn.LayerNorm(latent_dim))

        # K learnable queries
        self.queries = nn.Parameter(torch.randn(1, K, latent_dim) * 0.02)

        # Per-round layers
        self.cross_attns = nn.ModuleList()
        self.cross_norms = nn.ModuleList()
        self.cross_ffns  = nn.ModuleList()
        self.cross_fn    = nn.ModuleList()
        self.self_attns  = nn.ModuleList()
        self.self_norms  = nn.ModuleList()
        self.self_ffns   = nn.ModuleList()
        self.self_fn     = nn.ModuleList()

        for _ in range(n_rounds):
            self.cross_attns.append(nn.MultiheadAttention(latent_dim, n_heads, batch_first=True, dropout=0.2))
            self.cross_norms.append(nn.LayerNorm(latent_dim))
            self.cross_ffns.append(nn.Sequential(
                nn.Linear(latent_dim, latent_dim*2), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(latent_dim*2, latent_dim)))
            self.cross_fn.append(nn.LayerNorm(latent_dim))

            self.self_attns.append(nn.MultiheadAttention(latent_dim, n_heads, batch_first=True, dropout=0.2))
            self.self_norms.append(nn.LayerNorm(latent_dim))
            self.self_ffns.append(nn.Sequential(
                nn.Linear(latent_dim, latent_dim*2), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(latent_dim*2, latent_dim)))
            self.self_fn.append(nn.LayerNorm(latent_dim))

    def forward(self, x, mask=None):
        T_matrix = None
        if self.use_tnet: x, T_matrix = self.tnet(x, mask)

        h = self.phi(x)                                          # (B, N, D)
        q = self.queries.expand(x.size(0), -1, -1)               # (B, K, D)
        key_pad = ~mask if mask is not None else None

        for i in range(self.n_rounds):
            # cross-attention: queries ← points
            ca = self.cross_attns[i](self.cross_norms[i](q), h, h, key_padding_mask=key_pad)[0]
            q = q + ca
            q = q + self.cross_ffns[i](self.cross_fn[i](q))

            # self-attention: queries ↔ queries
            sa = self.self_attns[i](self.self_norms[i](q), q, q)[0]
            q = q + sa
            q = q + self.self_ffns[i](self.self_fn[i](q))

        return q, T_matrix                                       # (B, K, D)


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim), nn.LayerNorm(pred_dim), nn.GELU(),
            nn.Linear(pred_dim, dim))
    def forward(self, z): return self.net(z)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=False, n_rounds=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    indices = get_valid_indices(npz_file)
    dataset = JEPADataset(npz_file, indices)
    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    context_encoder = PerceiverEncoder(K=K, use_tnet=use_tnet, n_rounds=n_rounds).to(device)
    target_encoder  = PerceiverEncoder(K=K, use_tnet=use_tnet, n_rounds=n_rounds).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters(): p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)

    total_steps = epochs * len(train_loader)
    global_step = 0; best_val = float("inf")

    tnet_tag = "+T-Net" if use_tnet else ""
    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nPERCEIVER{tnet_tag} | K={K} | rounds={n_rounds} | {n_params:,} params | {device}")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        context_encoder.train(); predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx = ctx.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            z_context, T_ctx = context_encoder(ctx, ctx_mask)
            with torch.no_grad(): z_target, _ = target_encoder(tgt, tgt_mask)
            z_predicted = predictor(z_context)

            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)
            loss = F.smooth_l1_loss(pred_n, tgt_n.detach()) + F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            if use_tnet and T_ctx is not None: loss = loss + TNet.ortho_reg(T_ctx)
            if torch.isnan(loss): loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(list(context_encoder.parameters()) + list(predictor.parameters()), 1.0)
            optimizer.step(); scheduler.step()

            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)
                cos = F.cosine_similarity(pred_n.flatten(0,1), tgt_n.flatten(0,1), -1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item(); total_cos += cos.item()
            total_std += std.item(); n_batches += 1; global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        context_encoder.eval(); predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0
        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx = ctx.to(device); tgt = tgt.to(device)
                ctx_mask = ctx_mask.to(device); tgt_mask = tgt_mask.to(device)
                z_c, _ = context_encoder(ctx, ctx_mask)
                z_t, _ = target_encoder(tgt, tgt_mask)
                z_p = predictor(z_c)
                pn = F.normalize(z_p, -1, eps=1e-6); tn = F.normalize(z_t, -1, eps=1e-6)
                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0,1), tn.flatten(0,1), -1).mean()
                val_loss += vl.item(); val_cos += vc.item(); val_n += 1

        tl = total_loss/n_batches; tc = total_cos/n_batches; ts = total_std/n_batches
        vl = val_loss/val_n; vc = val_cos/val_n
        if vl < best_val: best_val = vl
        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

    torch.save({
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val, "K": K, "use_tnet": use_tnet, "n_rounds": n_rounds,
    }, f"/content/drive/MyDrive/Colab Notebooks/symbolic_data/Architecture_tests_17march/jepa_perceiver{'_tnet' if use_tnet else ''}.pt")
    print(f"\nDone! PERCEIVER{tnet_tag} — best val={best_val:.4f}")


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50, K=8, batch_size=1024, lr=2e-3, use_tnet=True, n_rounds=2,
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:04<00:00, 242.32it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

PERCEIVER+T-Net | K=8 | rounds=2 | 558,544 params | cuda
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:11<00:00,  3.44it/s, loss=0.9248, cos=0.063]


  loss=0.9316 cos=0.065 std=0.076 | val=52.7587 val_cos=0.011


...

  loss=0.9123 cos=0.920 std=0.088 | val=10.9960 val_cos=0.348


Epoch 49/50: 100%|██████████| 39/39 [00:10<00:00,  3.59it/s, loss=0.9124, cos=0.920]


  loss=0.9123 cos=0.920 std=0.088 | val=12.9279 val_cos=0.346


Epoch 50/50: 100%|██████████| 39/39 [00:10<00:00,  3.57it/s, loss=0.9123, cos=0.921]


  loss=0.9123 cos=0.920 std=0.088 | val=11.6658 val_cos=0.349

Done! PERCEIVER+T-Net — best val=10.8110


In [ ]:
"""
LM-JEPA: DeepSets + Slot Attention Encoder
Points compete for K slots via iterative softmax-over-slots assignment.
Each slot captures a different aspect of the equation.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ==========================================
# 1. DATASET
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []
    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue
    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices


class JEPADataset(Dataset):
    def __init__(self, npz_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.lengths = np.zeros(len(valid_indices), dtype=np.int32)

        kept = 0
        for real_idx in tqdm(valid_indices, desc="Pre-loading"):
            try:
                x_pts = np.array(full_data[f"X_{real_idx}"], dtype=np.float32)
                y_pts = np.array(full_data[f"y_{real_idx}"], dtype=np.float32).ravel()
            except: continue
            if x_pts.ndim == 1: x_pts = x_pts.reshape(-1, 1)
            n_p = min(x_pts.shape[0], len(y_pts), 500)
            n_v = min(x_pts.shape[1], 3)
            if n_p < 4: continue
            x, y = x_pts[:n_p, :n_v].copy(), y_pts[:n_p].copy()
            ok = np.isfinite(x).all(1) & np.isfinite(y)
            x, y = x[ok], y[ok]
            if len(y) < 4: continue
            for c in range(n_v):
                s = x[:, c].std() + 1e-8
                x[:, c] = (x[:, c] - x[:, c].mean()) / s
            ys = y.std() + 1e-8
            if ys < 1e-12: continue
            y = (y - y.mean()) / ys
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
            n_p = len(y)
            self.points[kept, :n_p, :n_v] = x
            self.points[kept, :n_p, 3] = y
            self.lengths[kept] = n_p
            kept += 1

        self.points = torch.from_numpy(self.points[:kept])
        self.lengths = torch.from_numpy(self.lengths[:kept])
        print(f"  Loaded {kept} equations, shape {self.points.shape}")

    def __len__(self): return len(self.points)

    def __getitem__(self, idx):
        n = int(self.lengths[idx])
        pts = self.points[idx, :n]
        perm = torch.randperm(n)
        nc = max(min(n // 2, n - 1), 1)
        ctx_pad = torch.zeros(250, 4); tgt_pad = torch.zeros(250, 4)
        ctx_mask = torch.zeros(250, dtype=torch.bool); tgt_mask = torch.zeros(250, dtype=torch.bool)
        ctx = pts[perm[:nc]]; tgt = pts[perm[nc:]]
        ctx_pad[:ctx.size(0)] = ctx; tgt_pad[:tgt.size(0)] = tgt
        ctx_mask[:ctx.size(0)] = True; tgt_mask[:tgt.size(0)] = True
        return ctx_pad, tgt_pad, ctx_mask, tgt_mask


# ==========================================
# 2. SLOT ATTENTION MODULE
# ==========================================
class SlotAttention(nn.Module):
    """
    Iterative slot competition.

    Each iteration:
      1. Compute affinity: slots (K, D) × points (N, D) → (K, N)
      2. Softmax OVER SLOTS (dim=0) → each point picks its slot
      3. Weighted mean of assigned points → slot update
      4. GRU gate controls how much each slot changes

    After n_iters, each slot owns a different subset of points.
    """

    def __init__(self, dim=128, K=8, n_iters=3, eps=1e-8):
        super().__init__()
        self.K = K
        self.n_iters = n_iters
        self.eps = eps
        self.dim = dim

        # Learned slot initializations
        self.slot_mu = nn.Parameter(torch.randn(1, K, dim) * 0.02)
        self.slot_log_sigma = nn.Parameter(torch.zeros(1, K, dim))

        # Projections for computing affinity
        self.project_q = nn.Linear(dim, dim)   # slots → queries
        self.project_k = nn.Linear(dim, dim)   # points → keys
        self.project_v = nn.Linear(dim, dim)   # points → values

        self.scale = dim ** -0.5

        # GRU for controlled slot updates
        self.gru = nn.GRUCell(dim, dim)

        # LayerNorms
        self.norm_slots = nn.LayerNorm(dim)
        self.norm_points = nn.LayerNorm(dim)

        # Per-slot MLP refinement after GRU
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, dim),
        )
        self.norm_mlp = nn.LayerNorm(dim)

    def forward(self, point_features, mask=None):
        """
        point_features: (B, N, D) — from DeepSets φ
        mask:           (B, N) bool — True = valid point
        Returns:        (B, K, D) — K slot vectors
        """
        B, N, D = point_features.shape

        # Initialize slots with learned distribution (adds stochasticity)
        mu = self.slot_mu.expand(B, -1, -1)
        sigma = self.slot_log_sigma.exp().expand(B, -1, -1)
        slots = mu + sigma * torch.randn_like(mu)              # (B, K, D)

        # Precompute keys and values from points (don't change across iterations)
        points_normed = self.norm_points(point_features)
        k = self.project_k(points_normed)                      # (B, N, D)
        v = self.project_v(points_normed)                      # (B, N, D)

        for _ in range(self.n_iters):
            slots_prev = slots

            # Step 1: compute affinity between slots and points
            slots_normed = self.norm_slots(slots)
            q = self.project_q(slots_normed)                   # (B, K, D)

            # (B, K, N) = (B, K, D) × (B, D, N)
            attn_logits = torch.bmm(q, k.transpose(1, 2)) * self.scale

            # Mask out padded points (set to -inf so softmax ignores them)
            if mask is not None:
                attn_logits = attn_logits.masked_fill(~mask[:, None, :], -1e9)

            # Step 2: softmax OVER SLOTS (dim=1) — points choose slots
            # This is the key difference from regular attention!
            # Regular attention: softmax over N points (dim=2)
            # Slot attention:    softmax over K slots  (dim=1)
            attn_weights = F.softmax(attn_logits, dim=1)       # (B, K, N)

            # Normalize by how many points each slot got (avoid division issues)
            slot_totals = attn_weights.sum(dim=2, keepdim=True) + self.eps  # (B, K, 1)
            attn_weights_normed = attn_weights / slot_totals

            # Step 3: weighted average of values for each slot
            updates = torch.bmm(attn_weights_normed, v)        # (B, K, D)

            # Step 4: GRU update — controlled slot change
            # GRUCell expects (batch*K, D) inputs
            slots = self.gru(
                updates.reshape(B * self.K, D),
                slots_prev.reshape(B * self.K, D),
            ).reshape(B, self.K, D)

            # MLP refinement with residual
            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots                                            # (B, K, D)


# ==========================================
# 3. ENCODER: DeepSets φ + Slot Attention
# ==========================================
class SlotEncoder(nn.Module):
    """
    Stage 1: DeepSets φ (per-point MLP, independent, no overfit risk)
    Stage 2: Slot Attention (iterative competition, captures inter-point info)
    """

    def __init__(self, input_dim=4, hidden_dim=128, latent_dim=128, K=8, n_iters=3):
        super().__init__()

        # DeepSets φ: shared MLP per point (same as your working baseline)
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        # Slot Attention: iterative competition → K slot vectors
        self.slot_attn = SlotAttention(
            dim=latent_dim, K=K, n_iters=n_iters)

    def forward(self, x, mask=None):
        """
        x:    (B, 250, 4)
        mask: (B, 250) bool
        Returns: (B, K, D)
        """
        # Stage 1: per-point features (independent, no inter-point info)
        h = self.phi(x)                                        # (B, 250, D)

        # Zero out features for padded points
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()

        # Stage 2: slot competition (captures inter-point structure)
        slots = self.slot_attn(h, mask)                        # (B, K, D)

        return slots


# ==========================================
# 4. PREDICTOR
# ==========================================
class Predictor(nn.Module):
    def __init__(self, dim=128, pred_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, pred_dim),
            nn.LayerNorm(pred_dim),
            nn.GELU(),
            nn.Linear(pred_dim, dim),
        )

    def forward(self, z):
        return self.net(z)                                     # (B, K, D) → (B, K, D)


# ==========================================
# 5. TRAINING
# ==========================================
def train_jepa(npz_file, epochs=50, K=8, batch_size=1024, lr=2e-3, n_iters=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Data
    indices = get_valid_indices(npz_file)
    dataset = JEPADataset(npz_file, indices)
    n_train = int(0.9 * len(dataset)); n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               pin_memory=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    # Models
    context_encoder = SlotEncoder(K=K, n_iters=n_iters).to(device)
    target_encoder  = SlotEncoder(K=K, n_iters=n_iters).to(device)
    predictor       = Predictor(dim=128, pred_dim=64).to(device)

    # Target encoder = EMA copy of context encoder, frozen
    target_encoder.load_state_dict(context_encoder.state_dict())
    for p in target_encoder.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=lr, weight_decay=0.05)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)

    total_steps = epochs * len(train_loader)
    global_step = 0
    best_val = float("inf")

    n_params = sum(p.numel() for p in context_encoder.parameters())
    print(f"\nSLOT ATTENTION | K={K} | iters={n_iters} | {n_params:,} params | {device}")
    print(f"Train: {n_train} | Val: {n_val} | Batch: {batch_size}\n")

    for epoch in range(1, epochs + 1):
        # ── Train ──
        context_encoder.train()
        predictor.train()
        total_loss, total_cos, total_std, n_batches = 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ctx, tgt, ctx_mask, tgt_mask in pbar:
            ctx      = ctx.to(device, non_blocking=True)
            tgt      = tgt.to(device, non_blocking=True)
            ctx_mask = ctx_mask.to(device, non_blocking=True)
            tgt_mask = tgt_mask.to(device, non_blocking=True)

            # Online encoder → context slots
            z_context = context_encoder(ctx, ctx_mask)          # (B, K, D)

            # EMA target encoder → target slots (no grad)
            with torch.no_grad():
                z_target = target_encoder(tgt, tgt_mask)        # (B, K, D)

            # Predictor: context slots → predicted target slots
            z_predicted = predictor(z_context)                  # (B, K, D)

            # Loss
            pred_n = F.normalize(z_predicted, dim=-1, eps=1e-6)
            tgt_n  = F.normalize(z_target, dim=-1, eps=1e-6)

            loss_inv = F.smooth_l1_loss(pred_n, tgt_n.detach())
            loss_var = F.relu(1.0 - pred_n.std(0).clamp(min=1e-6)).mean()
            loss = loss_inv + loss_var

            if torch.isnan(loss):
                loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(context_encoder.parameters()) + list(predictor.parameters()), 1.0)
            optimizer.step()
            scheduler.step()

            # EMA update (cosine schedule 0.996 → 1.0)
            tau = 1.0 - (1.0 - 0.996) * (math.cos(math.pi * global_step / total_steps) + 1) / 2
            with torch.no_grad():
                for p_t, p_o in zip(target_encoder.parameters(), context_encoder.parameters()):
                    p_t.data.lerp_(p_o.data, 1.0 - tau)

            # Metrics
            with torch.no_grad():
                cos = F.cosine_similarity(
                    pred_n.flatten(0, 1), tgt_n.flatten(0, 1), dim=-1).mean()
                std = pred_n.std(0).mean()

            total_loss += loss.item()
            total_cos  += cos.item()
            total_std  += std.item()
            n_batches  += 1
            global_step += 1

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "cos": f"{cos.item():.3f}"})

        # ── Validate ──
        context_encoder.eval()
        predictor.eval()
        val_loss, val_cos, val_n = 0, 0, 0

        with torch.no_grad():
            for ctx, tgt, ctx_mask, tgt_mask in val_loader:
                ctx      = ctx.to(device)
                tgt      = tgt.to(device)
                ctx_mask = ctx_mask.to(device)
                tgt_mask = tgt_mask.to(device)

                z_c = context_encoder(ctx, ctx_mask)
                z_t = target_encoder(tgt, tgt_mask)
                z_p = predictor(z_c)

                pn = F.normalize(z_p, dim=-1, eps=1e-6)
                tn = F.normalize(z_t, dim=-1, eps=1e-6)

                vl = F.smooth_l1_loss(pn, tn) + F.relu(1.0 - pn.std(0).clamp(1e-6)).mean()
                vc = F.cosine_similarity(pn.flatten(0, 1), tn.flatten(0, 1), dim=-1).mean()

                val_loss += vl.item()
                val_cos  += vc.item()
                val_n    += 1

        # ── Log ──
        tl = total_loss / n_batches
        tc = total_cos / n_batches
        ts = total_std / n_batches
        vl = val_loss / val_n
        vc = val_cos / val_n

        if vl < best_val:
            best_val = vl

        warn = " ⚠ COLLAPSE" if ts < 0.01 else ""
        print(f"  loss={tl:.4f} cos={tc:.3f} std={ts:.3f} | val={vl:.4f} val_cos={vc:.3f}{warn}")

    # Save at end
    torch.save({
        "context_encoder": context_encoder.state_dict(),
        "target_encoder": target_encoder.state_dict(),
        "predictor": predictor.state_dict(),
        "val_loss": best_val,
        "K": K, "n_iters": n_iters,
    }, "/content/drive/MyDrive/symbolic_data/final_last_2_march/jepa_slot_attention.pt")

    print(f"\nDone! SLOT ATTENTION — best val={best_val:.4f}")


# --- RUN ---
train_jepa(
    "/content/drive/MyDrive/symbolic_data/final_last_2_march/equations_50k_data.npz",
    epochs=50,
    K=8,
    batch_size=1024,
    lr=2e-3,
    n_iters=3,
)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [03:12<00:00, 232.29it/s]


  Loaded 44820 equations, shape torch.Size([44820, 500, 4])

SLOT ATTENTION | K=8 | iters=3 | 251,776 params | cuda
Train: 40338 | Val: 4482 | Batch: 1024



Epoch 1/50: 100%|██████████| 39/39 [00:07<00:00,  5.11it/s, loss=0.9234, cos=0.011]


  loss=0.9288 cos=0.018 std=0.079 | val=0.9233 val_cos=0.009


...

  loss=0.9125 cos=0.898 std=0.088 | val=0.9124 val_cos=0.898


Epoch 49/50: 100%|██████████| 39/39 [00:08<00:00,  4.79it/s, loss=0.9124, cos=0.898]


  loss=0.9125 cos=0.898 std=0.088 | val=0.9124 val_cos=0.898


Epoch 50/50: 100%|██████████| 39/39 [00:08<00:00,  4.81it/s, loss=0.9125, cos=0.899]


  loss=0.9125 cos=0.898 std=0.088 | val=0.9124 val_cos=0.898

Done! SLOT ATTENTION — best val=0.9124


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
